In [ ]:
import os, sys
sys.path.append('../src')
from FIT_python.data_split_and_summary.datasplit_and_summary_wraper import prepare_all_splits

# Daten-Splits sind bereits vorhanden, Aufruf bei Bedarf:
#prepare_all_splits()

In [ ]:
from FIT_python.Visualisations.plot_style import apply_style

apply_style()
import pandas as pd
from pathlib import Path
from FIT_python.pipeline_individual_id.evaluation import sequential_holdout_ids
from FIT_python.pipeline_individual_id.generate_trails_and_trailpairs import (
    generate_pairwise_comparisons_from_df,
)
from FIT_python.pipeline_individual_id.baseline_pipeline import DistanceBaseline
from FIT_python.pipeline_individual_id.supervised_umap_pipeline import run as run_umap

from FIT_python.pipeline_individual_id.siamese_pipeline import run as run_siamese
from FIT_python.pipeline_individual_id.evaluation import compute_confusion
import seaborn as sns, matplotlib.pyplot as plt
from FIT_python.data_split_and_summary.data_import_utils import (
    load_and_prep_df_individual,
    get_feature_cols,
)
from FIT_python.config import RESULTS_DATA_DIR, SPLITS_DIR


# %% 1) Basis-Pfade & Hyperparameter-Räume (unverändert)
splits_root = Path("data/splits/eurasian_otter")
models_root = (
    RESULTS_DATA_DIR
    / "eurasian_otter_random_search_standard_metrics"
    / "best_maj_test_pct"
)
output_root = Path("results/data/pairwise_comp_eurasian_otter")
output_root.mkdir(parents=True, exist_ok=True)

# 6) Loop über Spezies
species_dir = Path(SPLITS_DIR) / "eurasian_otter"
species = "eurasian_otter"
print(f"\n🔄 Processing species: {species}")

# 6.1) Train/Test laden & Fold entfernen
df = load_and_prep_df_individual(
    species, Path(SPLITS_DIR)
)  # falls hier bugs valid vales f und m checken

feature_cols = [
    c for c in df.columns if c.startswith(("dist", "ang", "t", "v")) and c != "trail"
]
ids = df["individual_id"].unique()
split = sequential_holdout_ids(ids, val_sizes=[3, 5, 9], n_iter=1, random_state=0)[0]
train_df = df[df["individual_id"].isin(split["train_ids"])]
val_df = df[df["individual_id"].isin(split["val_ids"])]
val_comps, _ = generate_pairwise_comparisons_from_df(
    val_df,
    id_col="individual_id",
    group_sizes=[ 2,3,5,7],
    n_repeats=10,
    mode="both",
    selfmatch_factor=2.0,
    n_folds=3,
    random_state=42,
    show_progress=True,
)

# In ein DataFrame überführen
df_val_comps = pd.DataFrame(val_comps)

In [ ]:
val_comps

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from FIT_python.pipeline_individual_id.baseline_pipeline import DistanceBaseline
from FIT_python.pipeline_individual_id.evaluation import compute_confusion

splits = sequential_holdout_ids(ids, val_sizes=[4, 8, 12], n_iter=3, random_state=0)

for split in splits:
    train_df = df[df["individual_id"].isin(split["train_ids"])]
    val_df   = df[df["individual_id"].isin(split["val_ids"])]
    val_comps, _ = generate_pairwise_comparisons_from_df(
        val_df, id_col="individual_id", group_sizes=[7],
        n_repeats=2, mode="both", selfmatch_factor=2.0,
        n_folds=2, random_state=42, show_progress=True,
    )

    res_base = DistanceBaseline.run(train_df, val_comps, feature_cols, val_df=val_df)
    Xb = res_base.filter(regex="^dist_")
    yb = res_base["same_individual"].map({"True": 1, "False": 0})

    cv = StratifiedKFold(n_splits=2, shuffle=True, random_state=42)
    lr = LogisticRegression(max_iter=200)
    y_pred = cross_val_predict(lr, Xb, yb, cv=cv)

    cm_base = compute_confusion(pd.DataFrame({"same_individual": yb, "pred": y_pred}))
    print(f"Iteration {split['iteration']}, n_val={split['n_val']}")
    display(cm_base)

In [ ]:
res_umap, emb_umap = run_umap(train_df, val_comps, feature_cols, val_df=val_df)
yu = res_umap["same_individual"].map({"True": 1, "False": 0})
pred_u = (res_umap["pred_same_proba"] > 0.5).astype(int)
res_umap["pred"] = (res_umap["pred_same_proba"] > 0.5).astype(int)
cm_umap = compute_confusion(res_umap, true_col="same_individual", pred_col="pred")

sns.heatmap(cm_umap, annot=True, fmt="d")
plt.show()

In [ ]:
# Siamese-Pipeline aktuell deaktiviert
res_siam = run_siamese(train_df, val_comps, feature_cols, val_df=val_df)
ys = pd.Series([c["same_individual"] for c in val_comps]).map({"True": 1, "False": 0})
pred_s = (pd.Series([r["pred_same"] for r in res_siam]) > 0.5).astype(int)
cm_siam = compute_confusion(pd.DataFrame({"same_individual": ys, "pred": pred_s}))
sns.heatmap(cm_siam, annot=True, fmt="d")
plt.show()